In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0"
%pip install -q --no-deps "bitsandbytes==0.48.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import gc
import json
import re
import time
import unicodedata
from decimal import Decimal
from pathlib import Path

import torch
import pandas as pd
import transformers
import bitsandbytes as bnb

from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoProcessor, GenerationConfig, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

In [5]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DOC_DIR = ROOT / "data/sroie"
DAY3_OUT = ROOT / "outputs/quantization"
DAY3_OUT.mkdir(parents=True, exist_ok=True)

manifest = json.loads((DOC_DIR / "manifest.json").read_text(encoding="utf-8"))

task_config = json.loads((DOC_DIR / "task_config.json").read_text(encoding="utf-8"))

FIELDS = tuple(task_config["fields"])
DOC_PROMPT = task_config["prompt"]
IMAGE_KWARGS = task_config["image_kwargs"]

lookup = {row["id"]: row for row in manifest["val"]}
probe_records = [lookup[doc_id] for doc_id in manifest["val_probe_ids"]]

ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bnb.__version__
}

print(ENV)
print("Validation receipts:", len(probe_records))

{'gpu': 'Tesla T4', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'transformers': '4.57.1', 'bitsandbytes': '0.48.2'}
Validation receipts: 20


**Checkpoints and NF4 recipe**

In [ ]:
CHECKPOINTS = {
    "2b": {
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "revision": "89644892e4d85e24eaac8bacfd4f463576704203"
    },
    "4b": {
        "model_id": "Qwen/Qwen3-VL-4B-Instruct",
        "revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17"
    }
}

SKIP_MODULES = ["visual", "lm_head"]
def nf4_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=SKIP_MODULES
    )

**prediction of one receipt**

In [7]:
def user_messages():
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": DOC_PROMPT}
        ]
    }]

@torch.inference_mode()
def predict_one(model, processor, generation_config, image_path):
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    
    prompt = processor.apply_chat_template(
        user_messages(),
        tokenize=False,
        add_generation_prompt=True
    )
    batch = processor(
        text=[prompt],
        images=[image],
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    input_length = batch["input_ids"].shape[1]
    merge = model.config.vision_config.spatial_merge_size
    visual_tokens = int(batch["image_grid_thw"].prod(dim=-1).sum().item()) // merge**2
    
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    
    output = model.generate(
        **batch,
        generation_config=generation_config,
        use_model_defaults=False,
        do_sample=False
    )
    
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak = torch.cuda.max_memory_allocated() / 1024**3
    new_ids = output[:, input_length:]

    return {
        "raw_answer": processor.batch_decode(
            new_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0],
        "input_tokens": input_length,
        "visual_tokens": visual_tokens,
        "new_tokens": new_ids.shape[1],
        "latency_s": seconds,
        "peak_gpu_GiB": peak
    }

In [8]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(
        unicodedata.normalize("NFKC", value).casefold().split()
    )
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?",
        text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [9]:
def score_receipt(raw, reference):
    try:
        json.loads(raw)
        raw_valid = True
    except ValueError:
        raw_valid = False
    
    text = raw.strip()
    lines = text.splitlines()
    wrapped = (
        len(lines) >= 3
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    )
    
    if wrapped:
        text = "\n".join(lines[1:-1])
    
    try:
        prediction = json.loads(text)
    except ValueError:
        prediction = None

    schema = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(
            isinstance(value, str) or value is None
            for value in prediction.values()
        )
    )
    correct = {}
    for field in FIELDS:
        if not schema:
            correct[field] = False
        elif prediction[field] is None or reference[field] is None:
            correct[field] = prediction[field] is reference[field]
        else:
            pred = normal(field, prediction[field])
            gold = normal(field, reference[field])
            correct[field] = pred is not None and pred == gold

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema,
        "used_fence_cleanup": wrapped,
        **{field + "_correct": value for field, value in correct.items()},
        "all_correct": all(correct.values())
    }

In [10]:
def run_setting(name, size, quantized):
    checkpoint = CHECKPOINTS[size]
    processor = AutoProcessor.from_pretrained(
        checkpoint["model_id"],
        revision=checkpoint["revision"],
        use_fast=False
    )
    
    extra = {}
    if quantized:
        extra["quantization_config"] = nf4_config()
    
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        checkpoint["model_id"],
        revision=checkpoint["revision"],
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
        **extra
    ).eval()
    
    quantized_names = [
        name for name, module in model.named_modules()
        if isinstance(module, bnb.nn.Linear4bit)
    ]
    
    print(name, "| 4-bit Linear layers:", len(quantized_names))
    if quantized_names:
        print("First converted layer:", quantized_names[0])
        
    generation_config = GenerationConfig(
        max_new_tokens=task_config["max_new_tokens"],
        do_sample=False,
        use_cache=True,
        bos_token_id=model.generation_config.bos_token_id,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id
    )
    
    run = {
        "source": "fresh",
        "run_info": {
            "setting": name,
            **checkpoint,
            "quantized": quantized,
            "compute_dtype": "torch.float16",
            "skip_modules": SKIP_MODULES if quantized else [],
            "environment": ENV,
            "task_config": task_config
        },
        "rows": []
    }

    _ = predict_one(model, processor, generation_config, probe_records[0]["image_path"])
    
    for record in tqdm(probe_records, desc=name):
        result = predict_one(model, processor, generation_config, record["image_path"])

        run["rows"].append({
            "id": record["id"],
            "reference": record["reference"],
            **result,
            **score_receipt(result["raw_answer"], record["reference"])
        })

        (DAY3_OUT / f"{name}.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")

    del model, processor
    return run